Den här notebooken hanterar:

1. Inläsning av `ads_clean.json`
2. Skapande av ett relativt balanserat testset med:
    - beskrivningar längre än 200 tecken (under den gränsen är lite eller ingen information relevant)
    - 3 grupper för beskrivningens längd (så att vi kontrollerar korta, medellånga och långa annonser)
    - 12 jobb anonser per månad (jan–aug)
3. Sparande av `test_set`, redo att annoteras
4. Sparande av `train_set`

In [1]:
import pandas as pd
import json

with open("../data/ads_clean.json", encoding="utf-8") as f:
    df_clean = pd.DataFrame(json.load(f))

print("Antal rader:", len(df_clean))

Antal rader: 4266


# Test set

In [2]:
# checking ads with short descriptions
shorty = df_clean[df_clean["description"].str.len() < 200]

print("Antal korta beskrivningar:", len(shorty))
shorty

Antal korta beskrivningar: 7


,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language
65,29380853,80f8379915bbc4167ad20b83e1201c429d25fbfb,IT-konsulter,Careerguide Media Group AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-14T15:36:27,2025-01-17T23:59:59,Vi söker IT-konsulter för kunds räkning. Kvali...,sv
102,29390154,01d64bec03d9d1ce3b1242e6d4762e13a0f3c09a,IT-konsulter,Careerguide Media Group AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-17T13:48:02,2025-01-20T23:59:59,Vi söker IT-konsulter för kunds räkning. Kvali...,sv
123,29399546,ad3f2999f896170128d538311b19884db0f38ee3,IT-konsulter,Careerguide Media Group AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-21T16:34:57,2025-01-24T23:59:59,Vi söker IT-konsulter för kunds räkning. Kvali...,sv
383,29507839,2b4e9a743cc424f8ae8cd51e6bb580f14564a4af,Cloud- och Infrastrukturspecialist,Liminity AB,Stockholm,Stockholms län,Infrastrukturarkitekt,2025-03-01T15:00:49,2025-08-18T23:59:59,Som specialist kommer du att underhålla modern...,sv
940,29692048,247284cb1c8df6e32e3f202f0d4648c7c81a7445,Associate Principal - Architecture,"LTIMindtree Sverige, filial till LTIMindtree Lim",Stockholm,Stockholms län,Systemarkitekt,2025-05-09T14:54:59,2025-05-23T23:59:59,Vi är öppna för att anpassa rollen eller arbet...,sv
955,29700087,0c31e09ca9e253bc58c62a116848930cca53053e,Principal - Architecture,"LTIMindtree Sverige, filial till LTIMindtree Lim",Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2025-05-13T15:36:39,2025-06-12T23:59:59,Vi är öppna för att anpassa rollen eller arbet...,sv
1534,29920986,13f2864b257c1cc1ce2658a423ec4a2afeba1939,Pega Business Architect,BR Technologies AB,Stockholm,Stockholms län,Enterprise Architect,2025-08-18T17:14:50,2025-09-01T23:59:59,Vi söker en erfaren Pega Business Architect so...,sv


In [3]:
# remove jobs with short descriptions
df_restricted = df_clean[df_clean["description"].str.len() >= 200]

print("Antal rader efter borttagning av annonser med korta beskrivningar:", len(df_restricted))

Antal rader efter borttagning av annonser med korta beskrivningar: 4259


In [4]:
# restrict to the analysis window: Jan-Aug of both years
df_restricted["published"] = pd.to_datetime(df_restricted["published"])

df_restricted["year"] = df_restricted["published"].dt.year
df_restricted["month"] = df_restricted["published"].dt.month
df_restricted = df_restricted[(df_restricted["month"] <= 8) & (df_restricted["year"].isin([2025, 2026]))].copy()

print("Antal rader efter urval av relevanta månader:", len(df_restricted))
df_restricted.head()

Antal rader efter urval av relevanta månader: 3273


,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language,year,month
0,29354331,d127a2a996c37ab43c33c5607ad285fde7cbdc50,Payment Platform Manager,Spiruetta AB,Stockholm,Stockholms län,Systemarkitekt,2025-01-02 09:24:37,2025-02-01T23:59:59,Spiruetta AB söker en erfaren Payment Platform...,sv,2025,1
1,29354585,26008c8a15f486a5b9163029fddd5d7fc3c03cb4,Projektledare,Randstad AB,Nacka,Stockholms län,Infrastrukturarkitekt,2025-01-02 10:15:14,2025-01-27T23:59:59,"Arbetsbeskrivning\nVA, Projekt och utredning\n...",sv,2025,1
2,29355065,80fee065cdecb056026d75c18c095d96f64d2ec6,Consultant Manager till ett av våra grymma Dyn...,Fellowmind Sweden AB,Stockholm,Stockholms län,"Verksamhetskonsult, IT",2025-01-02 11:39:23,2025-06-21T23:59:59,Att du dessutom får chansen att joina Sveriges...,sv,2025,1
3,29355172,ac28585698079a23f90482d302de7da2ccc8e1ed,Systemspecialist med inriktning IT-infrastrukt...,Trafikverket,Stockholm,Stockholms län,Systemanalytiker/Systemutredare,2025-01-02 12:07:23,2025-01-23T23:59:59,Är du intresserad av cybersäkerhet och vill va...,sv,2025,1
4,29355229,6bd58c9893a5bc98dc7adee09f49739847c31409,IT arkitekt / Lösningsarkitekt till energisekt...,Lexicare AB,Luleå,Norrbottens län,IT-arkitekt/Lösningsarkitekt,2025-01-02 12:17:37,2025-02-01T23:59:59,Uppdragsbeskrivning:\nLexicare AB söker en niv...,sv,2025,1


In [5]:
# length buckets
df_restricted["len_bucket"] = pd.qcut(df_restricted["description"].str.len(), 3, labels=["short", "medium", "long"])

# shuffles the ads
df_restricted = df_restricted.sample(frac=1, random_state=42)

len(df_restricted)

3273

In [6]:
# helpfunction to sample in a balanced way by lenght bucket
def sample_balanced(d, n_per_cell=4, seed=42):
    return d.groupby(["month", "len_bucket"], observed=True).sample(
        n=n_per_cell, random_state=seed
    )

In [7]:
# creating the test set and checking it
test_set = pd.concat([sample_balanced(df_restricted[df_restricted.year == y]) for y in (2025, 2026)])

print("Antal anonser:", len(test_set))
print("-" * 50)
print("\nAntal anonser per månad och år:")
print(test_set.groupby(["year", "month"]).size().unstack(0))
print("-" * 50)
print("\nAntal annonser per längdgrupp och år:")
print(test_set.groupby(["year", "len_bucket"], observed=True).size().unstack(0))

Antal anonser: 192
--------------------------------------------------

Antal anonser per månad och år:
year   2025  2026
month            
1        12    12
2        12    12
3        12    12
4        12    12
5        12    12
6        12    12
7        12    12
8        12    12
--------------------------------------------------

Antal annonser per längdgrupp och år:
year        2025  2026
len_bucket            
short         32    32
medium        32    32
long          32    32


In [8]:
# The test set only need "id" and "description", so I drop the rest of the cols
test_set = test_set[["id", "description"]].reset_index(drop=True)
test_set.head()

,id,description
0,29389255,Vi söker enData PlatformEngineer till uppdrag ...
1,29370869,"Tveka inte, ansök redan idag! Du kan enkelt sk..."
2,29392086,Business Architect (Pega)\nTillsvidareanställn...
3,29413273,Vill du arbeta i en teknologidriven miljö där ...
4,29377922,Techster Solutions befinner i dagsläget sig i ...


In [9]:
# Saving test set
with open("../data/test_set.json", "w", encoding="utf-8") as f:
    json.dump(test_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

# Training set

Träningssetet ska inte innehålla annonser från testsetet, så de tas bort och träningssetet sparas.

In [10]:
train_set = df_restricted[~df_restricted["id"].isin(test_set["id"])].reset_index(drop=True)

print("Antal annonser i hela den rensade dataframen:", len(df_restricted))
print("Antal annonser i test_set::", len(test_set))
print("Antal annonser i train_set::", len(train_set))
train_set.head()

Antal annonser i hela den rensade dataframen: 3273
Antal annonser i test_set:: 192
Antal annonser i train_set:: 3081


,id,version_id,title,employer,municipality,region,occupation,published,deadline,description,language,year,month,len_bucket
0,29556320,e5c7d20fce9469c7f36d1c498cf5b0810eb8b424,Business Consultant Dynamics 365 Supply Chain ...,Columbus Sweden AB,Stockholm,Stockholms län,"Affärskonsult, IT",2025-03-19 10:04:43,2025-09-05T23:59:59,Har du erfarenhet av Dynamics 365 och/eller AX...,sv,2025,3,medium
1,31266797,c8a51abb890ce471930df9475a2ddd149712fda6,Hälsoinformatiker för framtidens vårdinformation,REGION ÖSTERGÖTLAND,Linköping,Östergötlands län,"Verksamhetskonsult, IT",2026-07-10 17:08:31,2026-08-09T23:59:59,"Omvärlden förändras i snabb takt. Ny teknik, n...",sv,2026,7,long
2,31129893,55d1578cb4393e9a1348986791614a5b0d901200,Data Scientist till Innovativt Bolag,Friday Väst AB,Göteborg,Västra Götalands län,Data scientist,2026-06-02 19:12:40,2026-07-23T23:59:59,Är du en analytisk och lösningsorienterad pers...,sv,2026,6,medium
3,29479271,702683b5d60aa511ac15d0e7d09d0d3e5c32d8d0,Dynamics 365 Finance Consultant,Nexer AB,NaN,NaN,IT-arkitekt/Lösningsarkitekt,2025-02-19 12:38:38,2025-07-15T23:59:59,Vill du arbeta med den senaste tekniken och hj...,sv,2025,2,medium
4,31249545,80269b01bc0e2332a56bd070ef7ec34fe1ff06f2,"Enterprise arkitekt, Gävle/Uppsala/Stockholm",LANTMÄTERIET,Gävle,Gävleborgs län,Enterprise Architect,2026-07-06 01:35:55,2026-08-21T23:59:59,Lantmäteriet är den myndighet som kartlägger S...,sv,2026,7,long


In [11]:
# Dropping unused cols and converting timestamp back to string to be able to save in .json
train_set = train_set.drop(columns=["language", "year", "month", "len_bucket"], errors="ignore")
train_set["published"] = pd.to_datetime(train_set["published"]).dt.strftime("%Y-%m-%dT%H:%M:%S")
train_set.head()

,id,version_id,title,employer,municipality,region,occupation,published,deadline,description
0,29556320,e5c7d20fce9469c7f36d1c498cf5b0810eb8b424,Business Consultant Dynamics 365 Supply Chain ...,Columbus Sweden AB,Stockholm,Stockholms län,"Affärskonsult, IT",2025-03-19T10:04:43,2025-09-05T23:59:59,Har du erfarenhet av Dynamics 365 och/eller AX...
1,31266797,c8a51abb890ce471930df9475a2ddd149712fda6,Hälsoinformatiker för framtidens vårdinformation,REGION ÖSTERGÖTLAND,Linköping,Östergötlands län,"Verksamhetskonsult, IT",2026-07-10T17:08:31,2026-08-09T23:59:59,"Omvärlden förändras i snabb takt. Ny teknik, n..."
2,31129893,55d1578cb4393e9a1348986791614a5b0d901200,Data Scientist till Innovativt Bolag,Friday Väst AB,Göteborg,Västra Götalands län,Data scientist,2026-06-02T19:12:40,2026-07-23T23:59:59,Är du en analytisk och lösningsorienterad pers...
3,29479271,702683b5d60aa511ac15d0e7d09d0d3e5c32d8d0,Dynamics 365 Finance Consultant,Nexer AB,NaN,NaN,IT-arkitekt/Lösningsarkitekt,2025-02-19T12:38:38,2025-07-15T23:59:59,Vill du arbeta med den senaste tekniken och hj...
4,31249545,80269b01bc0e2332a56bd070ef7ec34fe1ff06f2,"Enterprise arkitekt, Gävle/Uppsala/Stockholm",LANTMÄTERIET,Gävle,Gävleborgs län,Enterprise Architect,2026-07-06T01:35:55,2026-08-21T23:59:59,Lantmäteriet är den myndighet som kartlägger S...


In [12]:
# Saving test set
with open("../data/train_set.json", "w", encoding="utf-8") as f:
    json.dump(train_set.to_dict(orient="records"), f, ensure_ascii=False, indent=2)